# Clase 09 — Tests de hipótesis III: categorías y comparaciones múltiples

**Curso:** Estadística Aplicada con Python · **Bloque 3:** Inferencia
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/09_tests_hipotesis_iii.md`

## Qué vas a ver en esta clase

- **Datos categóricos:** tablas de contingencia y **chi-cuadrado de independencia** (la tabla 2×2 a mano, frecuencias esperadas, grados de libertad), **V de Cramér**, residuos y bondad de ajuste.
- **La conexión con la clase 7:** en una tabla 2×2, **z² = χ²** (test de dos proporciones y chi-cuadrado son el mismo test).
- **Cuándo falla el chi-cuadrado:** la condición sobre las frecuencias esperadas y el **test exacto de Fisher**.
- **Comparaciones múltiples:** por qué 20 tests dan casi siempre un "hallazgo" falso, **Bonferroni** (FWER) y **Benjamini-Hochberg** (FDR).
- **Cómo aplicar las correcciones:** a los post-hoc de la clase 8 (Tukey) y a muchos chi-cuadrado sobre categorías o subgrupos distintos (p-hacking).
- **Una tabla para decidir** qué test usar (clases 7-9).

## Requisitos

- Clase 7 (p-value, errores de tipo I y II, test de dos proporciones) y clase 8 (ANOVA y post-hoc de Tukey).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Todos los datos (visitas por canal y conversiones, tiempos de preparación) son sintéticos: un *ejemplo ilustrativo*, no datos reales.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla**: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print("Librerías cargadas correctamente.")

---
## 1. Los datos: canal de captación × conversión

Una tienda en línea capta visitas por tres canales: email, redes sociales y buscador. Para cada visita sabemos el canal y si **compró** o no. Las dos variables son **categóricas**, así que no hay medias que comparar: hay que **contar** (*ejemplo ilustrativo*, datos sintéticos, 200 visitas por canal).

El resumen natural de dos variables categóricas es la **tabla de contingencia**: una celda por cada combinación de categorías, con el **conteo** de casos.

In [ ]:
canales = ["Email", "Redes sociales", "Buscador"]
compras = [48, 70, 42]                                   # ejemplo ilustrativo: compras de 200 visitas por canal
visitas = pd.DataFrame({
    "canal": np.repeat(canales, 200),
    "compra": np.concatenate([np.repeat(["sí", "no"], [k, 200 - k]) for k in compras])})   # 600 filas: una por visita
print(visitas.head(3).to_string(), "\n")

tabla = pd.crosstab(visitas["canal"], visitas["compra"]).loc[canales, ["sí", "no"]]
display(tabla)
print("Tasa de compra por canal (%):")
print((tabla["sí"] / tabla.sum(axis=1) * 100).round(1).to_string())

---
## 2. Chi-cuadrado de independencia: la tabla 2×2 a mano

Empezamos con dos canales y una tabla pequeña, para poder calcularlo todo a mano. H0: canal y compra son **independientes** (la tasa de compra es la misma). Bajo H0, la **frecuencia esperada** de cada celda es (total de la fila × total de la columna) / total general; el estadístico es **Σ (observada − esperada)² / esperada**, con (filas − 1)·(columnas − 1) grados de libertad.

In [ ]:
observada = np.array([[30, 70],
                      [45, 55]])          # filas: canal 1, canal 2; columnas: compra, no compra (100 visitas por canal)
tot_fila = observada.sum(axis=1, keepdims=True)
tot_col = observada.sum(axis=0, keepdims=True)
esperada = tot_fila * tot_col / observada.sum()
chi2_mano = ((observada - esperada) ** 2 / esperada).sum()
gl_chi = (observada.shape[0] - 1) * (observada.shape[1] - 1)
p_chi = stats.chi2.sf(chi2_mano, gl_chi)
print("Esperadas bajo H0:\n", esperada)
print(f"a mano: χ² = {chi2_mano:.3f}   gl = {gl_chi}   p = {p_chi:.4f}")

sin_corr = stats.chi2_contingency(observada, correction=False)
con_corr = stats.chi2_contingency(observada)                      # por defecto en 2×2: corrección de Yates
print(f"scipy sin corrección: χ² = {sin_corr.statistic:.3f}   p = {sin_corr.pvalue:.4f}")
print(f"scipy con corrección de Yates: χ² = {con_corr.statistic:.3f}   p = {con_corr.pvalue:.4f}")
assert np.isclose(chi2_mano, sin_corr.statistic) and np.isclose(p_chi, sin_corr.pvalue)

v_cramer = np.sqrt(chi2_mano / (observada.sum() * (min(observada.shape) - 1)))
print(f"V de Cramér = {v_cramer:.3f}")

**¿Qué observas?**

- Las esperadas son 37,5 y 62,5 en cada fila (la tasa global de compra es 75/200 = 37,5 %). Las observadas se apartan de ellas ±7,5.
- χ² = 4,80 con 1 gl → p ≈ 0,028 (sin corrección). Con la corrección de Yates sale 4,18 y p ≈ 0,041: para tablas 2×2 pequeñas es la versión más prudente. Di siempre cuál usas.
- V de Cramér ≈ 0,15: asociación presente pero débil. El p-value no dice cuán fuerte es.

*(Tu respuesta aquí.)*

---
## 3. La conexión con la clase 7: z² = χ²

En la clase 7 comparaste dos proporciones con `proportions_ztest` (varianza combinada). Una tabla 2×2 **es** una comparación de dos proporciones: "canal 1 frente a canal 2" en la columna "compra". Comprobemos que los dos tests son el mismo, también con el ejemplo de la clase 7 (244 de 4 000 frente a 200 de 4 000).

In [ ]:
z, p_z = proportions_ztest([30, 45], [100, 100])
print(f"z = {z:.4f}   p = {p_z:.4f}   z² = {z ** 2:.4f}   χ² ({'sin corrección'}) = {sin_corr.statistic:.4f}")
assert np.isclose(z ** 2, sin_corr.statistic) and np.isclose(p_z, sin_corr.pvalue)

# El ejemplo de la clase 7 (ejemplo ilustrativo): 244 de 4 000 frente a 200 de 4 000
z7, p7 = proportions_ztest([244, 200], [4000, 4000])
t7 = np.array([[244, 4000 - 244], [200, 4000 - 200]])
c7 = stats.chi2_contingency(t7, correction=False)
print(f"Clase 7: z = {z7:.3f}   p = {p7:.4f}   z² = {z7 ** 2:.3f}   χ² = {c7.statistic:.3f}   p(χ²) = {c7.pvalue:.4f}")
assert np.isclose(z7 ** 2, c7.statistic) and np.isclose(p7, c7.pvalue)

# ¿Y la corrección de Yates? No coincide con z² (es una corrección extra del chi-cuadrado)
print(f"Con Yates: χ² = {con_corr.statistic:.3f}  (≠ z² = {z ** 2:.3f})")

### Experimento guiado: ¿z² = χ² siempre?

Generamos 200 tablas 2×2 aleatorias (tamaños de grupo y tasas al azar) y comparamos, en cada una, z² con el χ² sin corrección.

In [ ]:
rng_z = np.random.default_rng(9)
dif_max = 0.0
for _ in range(200):
    n1, n2 = rng_z.integers(30, 500, size=2)
    a, b = rng_z.binomial(n1, rng_z.uniform(0.1, 0.5)), rng_z.binomial(n2, rng_z.uniform(0.1, 0.5))
    if a + b in (0, n1 + n2):
        continue
    zz, _ = proportions_ztest([a, b], [n1, n2])
    chi = stats.chi2_contingency(np.array([[a, n1 - a], [b, n2 - b]]), correction=False).statistic
    dif_max = max(dif_max, abs(zz ** 2 - chi))
print(f"Diferencia máxima |z² − χ²| en 200 tablas = {dif_max:.2e}")
assert dif_max < 1e-8

**¿Qué observas?**

- z² y χ² coinciden hasta el error de redondeo: **son el mismo test** (con la misma varianza combinada) y dan el mismo p-value de dos colas.
- Entonces, ¿para qué sirven los dos? El **z** conserva el **signo** (qué grupo convierte más), permite contrastes de **una cola** y da la diferencia de tasas con su IC. El **χ²** se generaliza a **más de dos categorías** (tablas 3×2, 4×5...), donde no hay una sola diferencia.

*(Tu respuesta aquí.)*

---
## 4. Tablas más grandes, residuos y bondad de ajuste

Con los **tres** canales la tabla es 3×2 y los grados de libertad son (3 − 1)·(2 − 1) = 2. Para ver **dónde** se concentra la asociación miramos los **residuos estandarizados** (observada − esperada) / √esperada: valores de módulo mayor que 2 destacan.

El **chi-cuadrado de bondad de ajuste** compara **una** variable categórica con una distribución esperada: por ejemplo, ¿se reparten por igual las incidencias entre tres turnos?

In [ ]:
obs3 = tabla.to_numpy()
r3 = stats.chi2_contingency(obs3)
print(f"χ² = {r3.statistic:.3f}   gl = {r3.dof}   p = {r3.pvalue:.4f}   V de Cramér = {np.sqrt(r3.statistic / (obs3.sum() * (min(obs3.shape) - 1))):.3f}")
print("Esperadas (todas ≥ 5):", r3.expected_freq[0].round(2), "mínima =", r3.expected_freq.min().round(2))
resid = (obs3 - r3.expected_freq) / np.sqrt(r3.expected_freq)
display(pd.DataFrame(resid.round(2), index=canales, columns=["compra", "no compra"]))

tasas = obs3[:, 0] / obs3.sum(axis=1)
plt.bar(canales, tasas * 100, color=["tab:blue", "tab:orange", "tab:green"])
plt.axhline(obs3[:, 0].sum() / obs3.sum() * 100, color="black", linestyle="--", label="tasa global")
plt.ylabel("% de visitas que compran"); plt.title("Conversión por canal"); plt.legend()
plt.show()

# Bondad de ajuste: incidencias por turno (esperado: reparto igual)
gof = stats.chisquare([30, 20, 10])
print(f"Bondad de ajuste: χ² = {gof.statistic:.2f}   p = {gof.pvalue:.4f}   (gl = 2, esperadas 20-20-20)")

**¿Qué observas?**

- p ≈ 0,004: las tasas de conversión difieren entre canales. V ≈ 0,14: asociación débil pero real.
- El residuo mayor está en "Redes sociales – compra" (2,28: más compras de las esperadas); el resto queda por debajo de 2 en valor absoluto.
- Igual que un ANOVA, el chi-cuadrado global **no dice qué canales difieren**: los residuos son una pista descriptiva, no un test post-hoc. Para comparar pares hará falta corregir (sección 8).
- La bondad de ajuste rechaza el reparto igual entre turnos (χ² = 10, p ≈ 0,007), pero no dice en qué turno se concentran las incidencias.

*(Tu respuesta aquí.)*

---
## 5. La condición del chi-cuadrado: frecuencias esperadas y test exacto de Fisher

El chi-cuadrado es una **aproximación** que funciona bien si las frecuencias **esperadas** (no las observadas) no son diminutas. **Regla práctica:** todas ≥ 5 (algunos textos aceptan que solo una pequeña parte esté entre 1 y 5, pero ninguna por debajo de 1). Si no se cumple, en una 2×2 se usa el **test exacto de Fisher**, que calcula el p-value exacto sin aproximar. Además el chi-cuadrado exige **conteos**, no porcentajes: con porcentajes el tamaño de la muestra desaparece y el resultado no tiene sentido.

In [ ]:
pequena = np.array([[3, 9],
                    [7, 2]])              # ejemplo ilustrativo: 21 casos
rc = stats.chi2_contingency(pequena)
print("Esperadas:\n", rc.expected_freq.round(2), "\nmínima esperada =", rc.expected_freq.min().round(2), "(< 5)")
print(f"χ² (con Yates): p = {rc.pvalue:.4f}")
print(f"Fisher exacto:      p = {stats.fisher_exact(pequena).pvalue:.4f}   odds ratio = {stats.fisher_exact(pequena).statistic:.3f}")

### Experimento guiado: la tasa exacta de falsos positivos con tablas pequeñas

Dos grupos de tamaño 5 y 25 con la **misma** tasa de éxito verdadera (5 %; H0 cierta). Calculamos, sin simulación, la **probabilidad exacta** de que cada test dé p < 0,05: sumamos la probabilidad binomial de cada tabla posible en la que el test rechaza. Un test bien calibrado debería dar 0,05 (o menos).

In [ ]:
n1_t, n2_t, p_ver = 5, 25, 0.05
pm1 = stats.binom.pmf(np.arange(n1_t + 1), n1_t, p_ver)
pm2 = stats.binom.pmf(np.arange(n2_t + 1), n2_t, p_ver)
fpr = {"χ² sin corrección": 0.0, "χ² con Yates": 0.0, "Fisher": 0.0}
for a in range(n1_t + 1):
    for b in range(n2_t + 1):
        if a + b == 0 or a + b == n1_t + n2_t:
            continue                                   # tabla sin variación: no se puede contrastar
        tab = np.array([[a, n1_t - a], [b, n2_t - b]])
        w = pm1[a] * pm2[b]
        if stats.chi2_contingency(tab, correction=False).pvalue < 0.05: fpr["χ² sin corrección"] += w
        if stats.chi2_contingency(tab, correction=True).pvalue < 0.05: fpr["χ² con Yates"] += w
        if stats.fisher_exact(tab).pvalue < 0.05: fpr["Fisher"] += w
for nombre, valor in fpr.items():
    print(f"{nombre:<28} tasa exacta de falsos positivos = {valor:.4f}")

**¿Qué observas?**

- Con esperadas diminutas, el chi-cuadrado sin corrección rechaza H0 cierta cerca del 7 % de las veces: **más** que el 5 % nominal. La aproximación no es fiable.
- Yates y Fisher son conservadores (≈ 0,7 %): protegen contra los falsos positivos a costa de potencia.
- Moraleja: comprueba las esperadas antes de fiarte del p-value del chi-cuadrado; con 2×2 pequeñas, Fisher.

*(Tu respuesta aquí.)*

---
## 6. Comparaciones múltiples: por qué 20 tests dan casi siempre un "hallazgo"

Con **un** test y H0 cierta, la probabilidad de un falso positivo es α = 0,05. Con **m** tests independientes y H0 cierta en todos, la probabilidad de **al menos un** falso positivo es **1 − (1 − α)^m**. Es la **tasa de error por familia** (FWER, *family-wise error rate*).

In [ ]:
alfa = 0.05
tabla_m = pd.DataFrame({"m": [1, 3, 5, 10, 15, 20, 50]})
tabla_m["P(al menos un falso positivo)"] = 1 - (1 - alfa) ** tabla_m["m"]
display(tabla_m.set_index("m").round(3))
print(f"m = 20: 1 − 0,95^20 = {1 - 0.95 ** 20:.4f}")

### Experimento guiado: 20 comparaciones entre grupos con la misma distribución

Repetimos 5 000 veces un "estudio" con **20 comparaciones**; en cada una los dos grupos (30 datos) vienen de la **misma** distribución normal, así que **todas las H0 son ciertas**. Contamos cuántos tests salen "significativos" a α = 0,05, sin corrección, con Bonferroni y con Benjamini-Hochberg.

In [ ]:
rng_mc = np.random.default_rng(42)
REPS, M = 5000, 20
datos_mc = rng_mc.normal(size=(REPS, M, 2, 30))
p_mc = stats.ttest_ind(datos_mc[:, :, 0, :], datos_mc[:, :, 1, :], axis=2).pvalue      # forma (REPS, M)

sin_corregir = (p_mc < 0.05)
bonf = (p_mc < 0.05 / M)
bh = np.array([multipletests(fila, alpha=0.05, method="fdr_bh")[0] for fila in p_mc])

print(f"Teoría: 1 − 0,95^20 = {1 - 0.95 ** M:.4f}")
print(f"Sin corregir: estudios con al menos un 'hallazgo' = {sin_corregir.any(axis=1).mean():.4f}")
print(f"Sin corregir: falsos positivos por estudio (media) = {sin_corregir.sum(axis=1).mean():.3f}")
print(f"Bonferroni (α/m = {0.05 / M:.4f}): estudios con algún falso positivo = {bonf.any(axis=1).mean():.4f}")
print(f"Benjamini-Hochberg: estudios con algún falso positivo = {bh.any(axis=1).mean():.4f}")

se = np.sqrt(0.6415 * (1 - 0.6415) / REPS)
assert abs(sin_corregir.any(axis=1).mean() - (1 - 0.95 ** M)) < 4 * se, "la simulación no coincide con la teoría"

plt.hist(sin_corregir.sum(axis=1), bins=np.arange(-0.5, 8.5, 1), edgecolor="white")
plt.title("Falsos positivos por estudio (20 tests, todas las H0 ciertas)")
plt.xlabel("número de tests 'significativos'"); plt.ylabel("número de estudios")
plt.show()

**¿Qué observas?**

- Sin corregir, ≈ 64 % de los estudios encuentra al menos un "hallazgo" aunque **no hay nada**: coincide con 1 − 0,95²⁰ = 0,642. En media aparece ≈ 1 falso positivo por estudio (20 · 0,05).
- Con Bonferroni y con Benjamini-Hochberg, cuando todas las H0 son ciertas la probabilidad de algún falso positivo baja a ≈ 5 %.
- Si el equipo mira 20 subgrupos y solo comunica el que "sale", está haciendo justo esto (*p-hacking*).

*(Tu respuesta aquí.)*

---
## 7. Bonferroni frente a FDR de Benjamini-Hochberg

- **Bonferroni:** rechaza solo si p ≤ α/m. Controla la **FWER**: la probabilidad de **cualquier** falso positivo. Muy conservador con muchos tests.
- **Benjamini-Hochberg (BH):** ordena los p-values y compara el i-ésimo con (i/m)·α; rechaza todos hasta el mayor que cumple la condición. Controla la **FDR** (*false discovery rate*): la **proporción esperada de falsos entre los descubrimientos**. Más potente; admite algún falso positivo.
- **Holm** (`method="holm"`) es un Bonferroni escalonado: controla también la FWER y **nunca** es menos potente.

Primero, con 8 p-values de ejemplo (ordenados); luego, un experimento con 20 tests de los cuales 5 tienen efecto real.

In [ ]:
pv = np.array([0.001, 0.008, 0.012, 0.021, 0.040, 0.049, 0.20, 0.50])
m = len(pv)
rech_b, p_aj_b, _, umbral_b = multipletests(pv, alpha=0.05, method="bonferroni")
rech_h, p_aj_h, _, _ = multipletests(pv, alpha=0.05, method="fdr_bh")
umbral_bh = np.arange(1, m + 1) / m * 0.05
display(pd.DataFrame({"p": pv, "umbral BH (i/m·α)": umbral_bh,
                      "p ajustado Bonferroni": p_aj_b, "rechaza Bonferroni": rech_b,
                      "p ajustado BH": p_aj_h, "rechaza BH": rech_h}).round(4))
print(f"Sin corregir rechazan {(pv < 0.05).sum()};  Bonferroni (α/m = {0.05 / m:.5f}) {rech_b.sum()};  BH {rech_h.sum()}")

In [ ]:
rng_fdr = np.random.default_rng(5)
REPS_F, M_F, N_REALES = 2000, 20, 5
acum = {"Sin corregir": [], "Bonferroni": [], "Benjamini-Hochberg": []}
reales = np.arange(M_F) < N_REALES
for _ in range(REPS_F):
    a = rng_fdr.normal(size=(M_F, 30))
    b = rng_fdr.normal(size=(M_F, 30))
    b[:N_REALES] += 0.9                                 # 5 comparaciones con efecto real de 0,9 σ
    pvs = stats.ttest_ind(a, b, axis=1).pvalue
    decisiones = [pvs < 0.05, multipletests(pvs, method="bonferroni")[0], multipletests(pvs, method="fdr_bh")[0]]
    for clave, rech in zip(acum, decisiones):
        acum[clave].append((rech.sum(), (rech & ~reales).sum(), (rech & reales).sum()))

filas = []
for clave, v in acum.items():
    v = np.array(v)
    filas.append({"método": clave,
                  "descubrimientos (media)": v[:, 0].mean(),
                  "falsos (media)": v[:, 1].mean(),
                  "potencia (efectos reales hallados)": v[:, 2].mean() / N_REALES,
                  "FDR (prop. de falsos entre hallazgos)": (v[:, 1] / np.maximum(v[:, 0], 1)).mean(),
                  "FWER (P de algún falso)": (v[:, 1] > 0).mean()})
display(pd.DataFrame(filas).set_index("método").round(3))

**¿Qué observas?**

- Con los 8 p-values: sin corregir se rechazan 6; Bonferroni solo 1; BH, 4. Las hipótesis con p entre 0,012 y 0,049 son las que separan a los métodos.
- Sin corregir: ≈ 5,4 descubrimientos por estudio (≈ 4,6 reales y ≈ 0,75 falsos); en el 54 % de los estudios hay al menos un falso positivo, y ≈ 12 % de lo que se comunica es falso.
- Bonferroni: casi ningún falso (FWER ≈ 4,6 %), pero solo encuentra ≈ 63 % de los efectos reales.
- BH: encuentra ≈ 81 % de los efectos reales y mantiene la FDR ≈ 3,8 % (por debajo de 5 %), aunque en ≈ 18 % de los estudios hay algún falso.
- **Toda corrección cuesta potencia.** Bonferroni si **un solo** falso positivo es costoso; FDR si exploras muchas hipótesis y verificarás después los candidatos.

*(Tu respuesta aquí.)*

---
## 8. Aplicar las correcciones: Tukey, pares de canales y muchos subgrupos

### 8.1 Post-hoc de la clase 8: Tukey ya corrige

Tres almacenes con 40 pedidos cada uno (*ejemplo ilustrativo*, construidos como en la clase 8). El **HSD de Tukey** ya controla la FWER de los tres pares: **no** hay que corregirlo otra vez. Si en cambio haces t-tests de Welch a pares, sí: el ajuste lo pones tú (Bonferroni o Holm).

In [ ]:
rng_datos = np.random.default_rng(42)
tiempo_a = rng_datos.normal(32, 6, 40)      # ejemplo ilustrativo (minutos)
tiempo_b = rng_datos.normal(33, 6, 40)
tiempo_c = rng_datos.normal(38, 6, 40)
pares = [(0, 1), (0, 2), (1, 2)]
etiq = ["A − B", "A − C", "B − C"]
gs = [tiempo_a, tiempo_b, tiempo_c]

tk = stats.tukey_hsd(*gs)
p_welch = np.array([stats.ttest_ind(gs[i], gs[j], equal_var=False).pvalue for i, j in pares])
tabla_pares = pd.DataFrame({
    "Tukey (ya ajustado)": [tk.pvalue[i, j] for i, j in pares],
    "Welch sin corregir": p_welch,
    "Welch + Bonferroni": multipletests(p_welch, method="bonferroni")[1],
    "Welch + Holm": multipletests(p_welch, method="holm")[1]}, index=etiq)
display(tabla_pares.round(4))

**¿Qué observas?**

- A − B no se distingue en ningún método; A − C y B − C sí. Las conclusiones coinciden: Tukey y Welch corregido cuentan la misma historia, con p ajustados de magnitud parecida.
- Lo que **no** debes hacer es aplicar Bonferroni **encima** de los p-values ajustados de Tukey: ya están ajustados y perderías potencia sin motivo.
- Holm nunca da un p ajustado mayor que Bonferroni.

*(Tu respuesta aquí.)*

### 8.2 Comparar pares de canales tras un chi-cuadrado significativo

El chi-cuadrado 3×2 de la sección 4 fue significativo. Para saber **qué canales** difieren hacemos un chi-cuadrado 2×2 por cada par (3 comparaciones) y corregimos con Bonferroni y Holm. La familia son los **3** pares.

In [ ]:
from itertools import combinations
pares_c = list(combinations(range(3), 2))
p_pares = np.array([stats.chi2_contingency(obs3[[i, j]], correction=False).pvalue for i, j in pares_c])
res_pc = pd.DataFrame({
    "tasas (%)": [f"{tasas[i] * 100:.1f} vs {tasas[j] * 100:.1f}" for i, j in pares_c],
    "p sin corregir": p_pares,
    "p Bonferroni": multipletests(p_pares, method="bonferroni")[1],
    "p Holm": multipletests(p_pares, method="holm")[1]}, index=[f"{canales[i]} − {canales[j]}" for i, j in pares_c])
display(res_pc.round(4))

**¿Qué observas?**

- Redes sociales se distingue de Buscador (p ajustado ≈ 0,005) y de Email (≈ 0,03-0,05 según el método); Email y Buscador no se distinguen (p ≈ 0,47).
- Con 3 comparaciones la corrección es suave (Bonferroni multiplica por 3); con 45 pares la potencia se resentiría mucho más.

*(Tu respuesta aquí.)*

### 8.3 Muchos chi-cuadrado sobre subgrupos: el p-hacking en acción

Ahora miramos la conversión de **email frente a redes sociales en 10 segmentos de clientes** (150 visitas por canal y segmento; datos sintéticos). Construimos el escenario: la tasa base es 20 %, y **solo en los segmentos 1 y 2** redes sociales convierte de verdad más (32 %); en los otros 8 no hay diferencia. Hacemos 10 chi-cuadrado 2×2 y vemos qué "sale" con y sin corrección. La semilla (148) es una réplica concreta e ilustrativa: en otras réplicas los números cambian.

In [ ]:
rng_seg = np.random.default_rng(148)
p_seg = []
for k in range(10):
    a = rng_seg.binomial(150, 0.20)                                  # canal 1: 20 % en todos los segmentos
    b = rng_seg.binomial(150, 0.32 if k < 2 else 0.20)               # canal 2: 32 % solo en los segmentos 1 y 2
    p_seg.append(stats.chi2_contingency(np.array([[a, 150 - a], [b, 150 - b]]), correction=False).pvalue)
p_seg = np.array(p_seg)

rech_sc = p_seg < 0.05
rech_bf = multipletests(p_seg, method="bonferroni")[0]
rech_bh = multipletests(p_seg, method="fdr_bh")[0]
res_seg = pd.DataFrame({"p": p_seg, "efecto real": np.arange(10) < 2,
                        "sin corregir": rech_sc, "Bonferroni": rech_bf, "BH": rech_bh},
                       index=[f"segmento {k + 1}" for k in range(10)])
display(res_seg.round(4))
real = np.arange(10) < 2
for nombre, r in [("sin corregir", rech_sc), ("Bonferroni", rech_bf), ("BH", rech_bh)]:
    print(f"{nombre:<18} descubrimientos = {r.sum()}   de ellos falsos = {(r & ~real).sum()}")
print(f"Prob. teórica de algún falso positivo con 8 nulos (sin corregir) = {1 - 0.95 ** 8:.3f}")

**¿Qué observas?**

- Sin corregir, 4 segmentos "salen" (p < 0,05): los dos reales **y dos falsos** (segmentos 3 y 8). Quien mirase solo los p-values sin corregir comunicaría "redes sociales convierte más en los segmentos 1, 2, 3 y 8", y la mitad sería ruido.
- Con Bonferroni y con BH quedan exactamente los dos segmentos reales.
- Pasa siempre: la probabilidad de algún falso con 8 nulos es 1 − 0,95⁸ = 0,337. **Decide la familia (los 10 segmentos) antes de mirar** y corrige por todo lo que miraste, no por lo que comunicas.

*(Tu respuesta aquí.)*

---
## 9. ¿Qué test elijo? Tabla de decisión (clases 7-9)

| Tipo de variable respuesta | Grupos y diseño | Test habitual | Si fallan los supuestos | Clase |
|---|---|---|---|---|
| Numérica | 1 grupo frente a un valor de referencia | t de una muestra | Wilcoxon de una muestra | 7 |
| Numérica | 2 grupos independientes | t de Welch | Mann-Whitney | 7 |
| Numérica | 2 medidas pareadas (antes/después) | t pareado | Wilcoxon de rangos con signo | 7 |
| Numérica | ≥ 3 grupos independientes | ANOVA + Tukey | Welch-ANOVA; Kruskal-Wallis | 8 |
| Categórica (sí/no) | 2 grupos independientes | z de dos proporciones (= χ² 2×2) | Fisher si esperadas < 5 | 7 y 9 |
| Categórica | ≥ 3 grupos, o variable con ≥ 3 categorías | χ² de independencia + V de Cramér | Agrupar categorías (Fisher solo en 2×2) | 9 |
| Categórica (una variable) | Reparto observado frente a esperado | χ² de bondad de ajuste | Agrupar categorías | 9 |
| Categórica (sí/no) | 2 medidas pareadas (mismos sujetos) | McNemar (idea, no lo vemos) | — | — |
| Cualquiera | Muchos tests a la vez | Corrección: Bonferroni/Holm (FWER) o BH (FDR) | Definir la familia antes | 9 |

> **Preguntas para decidir:** (1) ¿qué **tipo** de variable mido? (2) ¿**cuántos** grupos? (3) ¿**independientes o pareados**? (4) ¿se cumplen los **supuestos** (normalidad, varianzas, esperadas ≥ 5)? (5) ¿cuántos tests hago en **total**?

---
## Resumen de la clase

- **Chi-cuadrado de independencia** (`chi2_contingency`) sobre **conteos**: esperadas = fila × columna / total, gl = (f − 1)(c − 1); **V de Cramér** para la fuerza; residuos para ver dónde; bondad de ajuste con `chisquare`.
- **z² = χ²** en 2×2: el test de dos proporciones de la clase 7 y el chi-cuadrado sin corrección son el mismo test.
- **Esperadas ≥ 5**; si no, agrupa categorías o usa **Fisher** en 2×2. El chi-cuadrado sin corrección puede rechazar de más con esperadas diminutas.
- **Comparaciones múltiples:** P(algún falso positivo) = 1 − (1 − α)^m (0,642 con m = 20). **Bonferroni/Holm** controlan la FWER; **BH** controla la FDR y es más potente. Toda corrección cuesta potencia.
- **Aplicación:** Tukey ya está ajustado (no lo corrijas otra vez); con t-tests o chi-cuadrado a pares, ajusta tú; con muchos subgrupos, define la familia **antes** de mirar.
- **Tabla de decisión:** tipo de variable, número de grupos, independientes/pareados, supuestos.

**Siguiente paso:** el notebook de práctica de la clase 9 y, después, la clase 10 (correlación y regresión lineal). El cálculo de tamaños de muestra y el test A/B llegan en la clase 13.